# Construct pydantic model from text input

In [ ]:
from pydantic_ai import Agent

# this is quite fast model that also has tool use
agent = Agent(model="openrouter:arcee-ai/trinity-mini:free")

result = await agent.run("Give me an IT employee working in sweden, shortly")
result

In [ ]:
print(result.output)

In [ ]:
from pydantic import BaseModel, Field


class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int = Field(description="salary should be between 30k and 50k, the more senior the position, the higher salary", gt=30_000, lt=50_000)
    position: str


result = await agent.run(
    "Give me an IT employee working in sweden", output_type=EmployeeModel
)
result

In [ ]:
result.output

In [ ]:
result.output.name, result.output.age, result.output.salary

In [ ]:
result.output.model_dump()

In [ ]:
result = await agent.run(
    "Give me ten employees in AI and data engineering fields, so the roles can vary, salary must be between 30000 and 50000",
    output_type=list[EmployeeModel],
)
result

In [ ]:
result.output

## CV model - a more complex and nested model

In [ ]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description: str
    start_year: int
    end_year: int


class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description: str
    start_year: int
    end_year: int


class CvModel(BaseModel):
    name: str
    age: int
    experiences: list[ExperienceModel]
    educations: list[EducationModel]


result = await agent.run(
    "Create 5 fake persons that is applying for a data engineering job. They should have 2-5 previous job experiences",
    output_type=list[CvModel],
)



In [ ]:
result.output[-1]

In [ ]:
result.output[-1].name

In [ ]:
result.output[-1].experiences[0].title, result.output[-1].experiences[1].title

## (optional) Postprocessing - load into duckdb and unnesting

This part is optional, but a way to unnest the data and store it could be to use dlt to load the data into duckdb, followed by joining and unnesting.

Other approach could be to store into nosql such as mongodb.

In [ ]:
import dlt

pipeline = dlt.pipeline(
    pipeline_name="cv_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging",
)

info = pipeline.run(
    data=[result.output], loader_file_format="jsonl", table_name="cv_entries"
)

print(info)


In [ ]:
import duckdb 

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc;").df()
    cv_entries = conn.sql("FROM staging.cv_entries").df()
    educations = conn.sql("FROM staging.cv_entries__educations").df()
    experiences = conn.sql("FROM staging.cv_entries__experiences").df()

desc

In [ ]:
cv_entries

In [ ]:
educations

In [ ]:
experiences

In [ ]:
duckdb.sql("""
    SELECT 
        cv.name, 
        cv.age, 
        ex.company,
        ex.description AS experience_description,
        ex.start_year AS experience_start_year,
        ex.end_year AS experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year AS education_start_year,
        e.end_year AS education_end_year
    FROM cv_entries cv
    LEFT JOIN educations e ON cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experiences ex ON cv._dlt_id = ex._dlt_parent_id
    

""").df()